<a href="https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sobaannr/FlyRank-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Lane: Refresh / Content Opportunity Scoring**

I'm picking this lane because I already have direct evidence it works, from the
Week 1-2 starter pipeline: a learned model beat the hand-written baseline rule at
picking which pages to review first — Precision@50 went from 0.240 (baseline) to
0.740 (random forest), per `outputs/model_report.md`. That's roughly 12 correct
picks out of the baseline's top 50 versus 37 correct picks out of the model's top
50. I want to build on that gap directly: a ranked review queue with reason codes
a human can inspect, rather than just a correlation report (Lane 1) or a
clustering exercise (Lane 3) without a clear archetype hypothesis yet.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print(f"Rows: {len(df)}, columns: {len(df.columns)}")
print(f"Has trend_direction: {'trend_direction' in df.columns}")
print(f"Has impressions_90d: {'impressions_90d' in df.columns}")

Rows: 30000, columns: 44
Has trend_direction: True
Has impressions_90d: True


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Unit of analysis:** one page (`content_id`) — matches the grain of
`dim_content` and the starter CSV.

**Decision this improves:** which pages a content reviewer looks at first out of
a much larger inventory, when only a small fraction can be manually reviewed each
cycle.

**Who acts on it:** a content reviewer, using the ranked list plus reason codes
as a starting point — not a final verdict.

**Cost of a wrong recommendation:**
- False positive (page ranked high, didn't need review): wastes limited reviewer
  time — a real cost when capacity is small relative to total inventory.
- False negative (a genuinely declining, high-demand page ranked low or missed):
  worse, because a page with real traffic behind it keeps losing visibility
  unnoticed until the next review cycle.

Because these costs aren't symmetric, Precision@K (K matched to real review
capacity) is the right metric here — not plain accuracy.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

review_capacity = 50
pct_covered = review_capacity / len(df) * 100
print(f"Total pages: {len(df)}")
print(f"A team reviewing top {review_capacity}/cycle covers {pct_covered:.2f}% of inventory")
print("Ranking quality matters more than raw review volume, since capacity is such a small slice.")

Total pages: 30000
A team reviewing top 50/cycle covers 0.17% of inventory
Ranking quality matters more than raw review volume, since capacity is such a small slice.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

Loading the starter dataset and pulling numbers that show this lane is worth
pursuing:

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

declining = df[df["trend_direction"] == "down"]
pct_declining = len(declining) / len(df) * 100
pct_impressions_at_risk = declining["impressions_90d"].sum() / df["impressions_90d"].sum() * 100

print(f"Pages flagged declining: {len(declining)} of {len(df)} ({pct_declining:.1f}%)")
print(f"Share of total impression volume sitting in declining pages: {pct_impressions_at_risk:.1f}%")

print("\nVerified pipeline results (outputs/model_report.md):")
print("baseline rules     Precision@50: 0.240  (~12 of 50 correct)")
print("random forest       Precision@50: 0.740  (~37 of 50 correct)")

print(f"\nReview capacity covers only {pct_covered:.2f}% of the {len(df)}-page inventory per cycle")


Pages flagged declining: 16262 of 30000 (54.2%)
Share of total impression volume sitting in declining pages: 51.3%

Verified pipeline results (outputs/model_report.md):
baseline rules     Precision@50: 0.240  (~12 of 50 correct)
random forest       Precision@50: 0.740  (~37 of 50 correct)

Review capacity covers only 0.17% of the 30000-page inventory per cycle


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*


**I can claim:** observed, historical association between signals (impressions,
position, freshness, trend) and pages later flagged as declining or under-
performing; that a ranked queue built from these signals would have surfaced more
true declining/opportunity pages in its top-K than a hand-rule did, under
client-holdout validation, on this dataset; directional, decision-support
language like "this page shows evidence consistent with decline and deserves
review."

**I cannot claim:** that refreshing a page causes recovery (that needs an actual
experiment, not this data); that I've discovered or reverse-engineered anything
about Google's ranking algorithm; that the starter's `is_declining_label`
(a same-window bucket, not a future outcome) is a strong target — it's a
beginner proxy, and my capstone should move toward a true future-window label;
anything that could re-identify a real client, domain, URL, or query.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

unsafe_keywords = ["url", "domain", "client_name", "query", "title"]
flagged_cols = [c for c in df.columns if any(k in c.lower() for k in unsafe_keywords)]
print(f"Columns matching unsafe-field keywords: {flagged_cols if flagged_cols else 'None found'}")

Columns matching unsafe-field keywords: None found


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.